In [1]:
!pip install langsmith -q


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\whizz\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ══════════════════════════════════════════════════════════
#  PASO 0: Instalación de dependencias
# ══════════════════════════════════════════════════════════
import sys
!{sys.executable} -m pip install faiss-cpu langchain-community langchain-openai langchain langchain-text-splitters python-dotenv -q
print("✅ Dependencias instaladas")

✅ Dependencias instaladas



[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ══════════════════════════════════════════════════════════
#  PASO 1: Configuración del modelo
# ══════════════════════════════════════════════════════════
import os
#os.environ["LANGCHAIN_TRACING_V2"] = "false"
#os.environ["LANGCHAIN_API_KEY"] = "dummy"

from dotenv import load_dotenv
load_dotenv()
from langsmith import Client, traceable

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "RAG-Vulcanizadora")

try:
    ls_client = Client()
    print("✅ LangSmith conectado")
except Exception as e:
    print(f"⚠️ Error: {e}")
    ls_client = None
    
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage

token = os.environ["GITHUB_TOKEN"]
endpoint = "https://models.github.ai/inference"
model_name = "openai/gpt-4.1"

os.environ["OPENAI_API_KEY"] = token
os.environ["OPENAI_BASE_URL"] = endpoint

llm = ChatOpenAI(
    base_url=endpoint,
    api_key=token,
    model=model_name,
    temperature=0.1
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url=endpoint,
    api_key=token
)

print("✅ Modelo de chat configurado:", model_name)
print("✅ Modelo de embeddings configurado: text-embedding-3-small")

✅ LangSmith conectado
✅ Modelo de chat configurado: openai/gpt-4.1
✅ Modelo de embeddings configurado: text-embedding-3-small


In [4]:
# ══════════════════════════════════════════════════════════
#  PASO 2: Base de conocimiento de la Vulcanizadora
# ══════════════════════════════════════════════════════════

documentos_vulcanizadora = [
    """DIAGNÓSTICO - ALAMBRES SALIDOS Y FORMA DE HUEVO:
    Cuando un neumático presenta alambres expuestos junto con deformación en forma de huevo,
    indica daño estructural severo en la carcasa. Este daño es irreparable.
    VEREDICTO: Cambio obligatorio. No se puede parchar ni reparar.
    URGENCIA: Alta. Riesgo inminente de reventón.""",

    """DIAGNÓSTICO - CLAVO O TORNILLO EN LA BANDA DE RODADURA:
    Un objeto metálico en la banda central generalmente es reparable con parche interno.
    CONDICIONES: El daño debe estar en la zona central, diámetro menor a 6mm,
    y el neumático no debe haber rodado desinflado más de 500 metros.
    VEREDICTO: Se puede parchar. Si el clavo está en el flanco, cambio obligatorio.
    URGENCIA: Media.""",

    """DIAGNÓSTICO - HERNIA O BURBUJA EN EL FLANCO:
    Una hernia en el flanco indica capas internas rotas por golpe fuerte o sobreinflado.
    VEREDICTO: Cambio obligatorio. Los flancos no se pueden parchar.
    URGENCIA: Alta. Puede reventar sin previo aviso.""",

    """DIAGNÓSTICO - CORTE EN EL FLANCO LATERAL:
    Cortes en el flanco lateral no son reparables si superan 5mm o comprometen la estructura.
    VEREDICTO: Cambio obligatorio para cortes profundos.
    URGENCIA: Alta.""",

    """DIAGNÓSTICO - DESGASTE IRREGULAR O EXCESIVO:
    Cuando el dibujo llega al indicador de desgaste (1.6mm) el cambio es obligatorio.
    - Desgaste central: exceso de presión.
    - Desgaste en bordes: falta de presión.
    - Desgaste lateral: problema de alineación.
    - Desgaste en parches: problema de balanceo.
    URGENCIA: Media a alta según nivel de desgaste.""",

    """SERVICIOS Y PRECIOS - VULCANIZADORA:
    REPARACIONES:
    - Parche interno vulcanizado (pinchazo estándar): $6.000
    - Parche grande (daño mayor): $9.000
    - Reparación de válvula: $3.000
    - Vulcanizado en caliente: $12.000

    MONTAJE Y DESMONTAJE:
    - Desmontaje y montaje de neumático (por unidad): $5.000
    - Montaje completo con aro (por unidad): $4.000

    BALANCEO Y ALINEACIÓN:
    - Balanceo dinámico (por rueda): $4.500
    - Balanceo de las 4 ruedas: $15.000
    - Alineación 2 ruedas delanteras: $18.000
    - Alineación completa 4 ruedas: $25.000

    INFLADO:
    - Inflado con nitrógeno (por rueda): $2.000
    - Revisión y ajuste de presión: Gratis""",

    """PROMOCIONES Y GARANTÍAS - VULCANIZADORA:
    PROMOCIONES:
    - Orden mayor a $100.000: 15% descuento en el servicio más caro.
    - Cambio de 4 neumáticos: montaje + balanceo incluido sin costo.
    - Clientes frecuentes (más de 3 visitas): 10% descuento en reparaciones.

    GARANTÍAS:
    - Parches internos: 6 meses o 10.000 km.
    - Balanceo: 3 meses o 5.000 km.
    - Montaje y desmontaje: garantía inmediata en taller.""",
]

print(f"📚 Base de conocimiento cargada: {len(documentos_vulcanizadora)} documentos")
for i, doc in enumerate(documentos_vulcanizadora, 1):
    primera_linea = doc.strip().split('\n')[0][:70]
    print(f"  {i}. {primera_linea}...")

📚 Base de conocimiento cargada: 7 documentos
  1. DIAGNÓSTICO - ALAMBRES SALIDOS Y FORMA DE HUEVO:...
  2. DIAGNÓSTICO - CLAVO O TORNILLO EN LA BANDA DE RODADURA:...
  3. DIAGNÓSTICO - HERNIA O BURBUJA EN EL FLANCO:...
  4. DIAGNÓSTICO - CORTE EN EL FLANCO LATERAL:...
  5. DIAGNÓSTICO - DESGASTE IRREGULAR O EXCESIVO:...
  6. SERVICIOS Y PRECIOS - VULCANIZADORA:...
  7. PROMOCIONES Y GARANTÍAS - VULCANIZADORA:...


In [5]:
!pip install langchain-text-splitters -q
print("✅ langchain-text-splitters instalado")

✅ langchain-text-splitters instalado



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\whizz\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
# ══════════════════════════════════════════════════════════
#  PASO 3: Chunking
# ══════════════════════════════════════════════════════════
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    length_function=len
)

chunks = text_splitter.create_documents(documentos_vulcanizadora)

print(f"📄 Documentos originales: {len(documentos_vulcanizadora)}")
print(f"🧩 Chunks generados:      {len(chunks)}")
print(f"📊 Promedio chars/chunk:  {sum(len(c.page_content) for c in chunks) // len(chunks)}")

📄 Documentos originales: 7
🧩 Chunks generados:      9
📊 Promedio chars/chunk:  295


In [7]:
# ══════════════════════════════════════════════════════════
#  PASO 4: Base de datos vectorial FAISS
# ══════════════════════════════════════════════════════════
from langchain_community.vectorstores import FAISS

print("⏳ Generando embeddings y creando base vectorial...")

try:
    vector_db = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )
    print("✅ Base de datos vectorial FAISS creada exitosamente")
    print(f"📊 Vectores almacenados: {vector_db.index.ntotal}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 Verificar que GITHUB_TOKEN esté configurado correctamente")

⏳ Generando embeddings y creando base vectorial...
✅ Base de datos vectorial FAISS creada exitosamente
📊 Vectores almacenados: 9


In [8]:
# ══════════════════════════════════════════════════════════
#  PASO 5: Pipeline RAG completo
# ══════════════════════════════════════════════════════════
@traceable(name="RAG-Vulcanizadora-Pipeline")
def rag_vulcanizadora(pregunta, k=3, mostrar_chunks=False):
    print(f"\n{'='*60}")
    print(f"🙋 Cliente: {pregunta}")
    print(f"{'='*60}")

    # FASE 1: RETRIEVAL
    retriever = vector_db.as_retriever(search_kwargs={"k": k})
    chunks_relevantes = retriever.invoke(pregunta)

    if mostrar_chunks:
        print(f"\n📋 Chunks recuperados ({len(chunks_relevantes)}):")
        for i, chunk in enumerate(chunks_relevantes, 1):
            print(f"  [{i}] {chunk.page_content[:120]}...")

    # FASE 2: GENERATION
    contexto = "\n\n".join([c.page_content for c in chunks_relevantes])

    prompt_rag = f"""Eres un profesional experto en vulcanizado y neumáticos.

INSTRUCCIONES:
- Responde ÚNICAMENTE basándote en el contexto proporcionado.
- Si la respuesta no está en el contexto, dilo claramente.
- Usa el siguiente formato:
  - Diagnóstico: [análisis del problema]
  - Recomendación: [acción a tomar]
  - Precio estimado: [si hay información de precios]
  - Urgencia: [nivel de urgencia]

CONTEXTO DEL TALLER:
{contexto}

CONSULTA DEL CLIENTE:
{pregunta}

RESPUESTA PROFESIONAL:"""

    response = llm.invoke([HumanMessage(content=prompt_rag)])

    print(f"\n🔧 Vulcanizador (RAG):")
    print(response.content)
    print(f"{'='*60}\n")

    return response.content

print("✅ Función RAG definida y lista")

✅ Función RAG definida y lista


In [9]:
# ══════════════════════════════════════════════════════════
#  PASO 6: Pruebas del sistema RAG
# ══════════════════════════════════════════════════════════

# Prueba 1: caso clásico
rag_vulcanizadora(
    "Tengo un neumático con los alambres salidos y forma de huevo, ¿lo parcho o lo cambio?",
    mostrar_chunks=True
)


🙋 Cliente: Tengo un neumático con los alambres salidos y forma de huevo, ¿lo parcho o lo cambio?

📋 Chunks recuperados (3):
  [1] DIAGNÓSTICO - ALAMBRES SALIDOS Y FORMA DE HUEVO:
    Cuando un neumático presenta alambres expuestos junto con deformaci...
  [2] DIAGNÓSTICO - CLAVO O TORNILLO EN LA BANDA DE RODADURA:
    Un objeto metálico en la banda central generalmente es repar...
  [3] BALANCEO Y ALINEACIÓN:
    - Balanceo dinámico (por rueda): $4.500
    - Balanceo de las 4 ruedas: $15.000
    - Alineac...

🔧 Vulcanizador (RAG):
Diagnóstico: El neumático presenta alambres expuestos y deformación en forma de huevo, lo que indica daño estructural severo en la carcasa. Este daño es irreparable.

Recomendación: Cambio obligatorio del neumático. No se puede parchar ni reparar.

Precio estimado: No hay información de precios para el cambio de neumático en el contexto proporcionado.

Urgencia: Alta. Existe riesgo inminente de reventón.



'Diagnóstico: El neumático presenta alambres expuestos y deformación en forma de huevo, lo que indica daño estructural severo en la carcasa. Este daño es irreparable.\n\nRecomendación: Cambio obligatorio del neumático. No se puede parchar ni reparar.\n\nPrecio estimado: No hay información de precios para el cambio de neumático en el contexto proporcionado.\n\nUrgencia: Alta. Existe riesgo inminente de reventón.'

In [10]:
# Prueba 2: precio (info específica del taller)
rag_vulcanizadora("¿Cuánto cuesta un parche y el balanceo de las 4 ruedas?")


🙋 Cliente: ¿Cuánto cuesta un parche y el balanceo de las 4 ruedas?

🔧 Vulcanizador (RAG):
Diagnóstico: El cliente solicita el precio de un parche (reparación de pinchazo estándar) y el balanceo de las 4 ruedas.

Recomendación: Realizar el parche interno vulcanizado para reparar el pinchazo y efectuar el balanceo dinámico de las 4 ruedas para asegurar un correcto funcionamiento y evitar vibraciones.

Precio estimado:  
- Parche interno vulcanizado: $6.000  
- Balanceo de las 4 ruedas: $15.000  
Total: $21.000

Urgencia: Media. Es recomendable reparar el pinchazo y balancear las ruedas lo antes posible para evitar daños adicionales y garantizar seguridad.



'Diagnóstico: El cliente solicita el precio de un parche (reparación de pinchazo estándar) y el balanceo de las 4 ruedas.\n\nRecomendación: Realizar el parche interno vulcanizado para reparar el pinchazo y efectuar el balanceo dinámico de las 4 ruedas para asegurar un correcto funcionamiento y evitar vibraciones.\n\nPrecio estimado:  \n- Parche interno vulcanizado: $6.000  \n- Balanceo de las 4 ruedas: $15.000  \nTotal: $21.000\n\nUrgencia: Media. Es recomendable reparar el pinchazo y balancear las ruedas lo antes posible para evitar daños adicionales y garantizar seguridad.'

In [11]:
# Prueba 3: fuera del contexto (debe decir que no sabe)
rag_vulcanizadora("¿Cuánto cuesta la revisión técnica del auto?")


🙋 Cliente: ¿Cuánto cuesta la revisión técnica del auto?

🔧 Vulcanizador (RAG):
Diagnóstico: No hay información en el contexto proporcionado sobre el costo de la revisión técnica del auto.

Recomendación: Consulte en un centro autorizado de revisión técnica, ya que este taller solo ofrece servicios relacionados con neumáticos y vulcanizado.

Precio estimado: No disponible en el contexto.

Urgencia: No aplica.



'Diagnóstico: No hay información en el contexto proporcionado sobre el costo de la revisión técnica del auto.\n\nRecomendación: Consulte en un centro autorizado de revisión técnica, ya que este taller solo ofrece servicios relacionados con neumáticos y vulcanizado.\n\nPrecio estimado: No disponible en el contexto.\n\nUrgencia: No aplica.'

In [12]:
respuesta = rag_vulcanizadora("¿Cuál es el problema más común en neumáticos?")
print(respuesta)


🙋 Cliente: ¿Cuál es el problema más común en neumáticos?

🔧 Vulcanizador (RAG):
Diagnóstico: El problema más común en neumáticos es el desgaste irregular o excesivo. Este puede manifestarse en diferentes formas, como desgaste central (por exceso de presión), desgaste en bordes (por falta de presión), desgaste lateral (por problemas de alineación) o desgaste en parches (por problemas de balanceo). Cuando el dibujo llega al indicador de desgaste (1.6mm), el cambio es obligatorio.

Recomendación: Revisar periódicamente la presión, alineación y balanceo de los neumáticos para prevenir desgaste irregular. Cambiar el neumático si el desgaste llega al indicador.

Precio estimado: No hay información de precios en el contexto.

Urgencia: Media a alta, dependiendo del nivel de desgaste.

Diagnóstico: El problema más común en neumáticos es el desgaste irregular o excesivo. Este puede manifestarse en diferentes formas, como desgaste central (por exceso de presión), desgaste en bordes (por falta d

In [13]:
respuesta = rag_vulcanizadora("¿Cuanto seria un cambio e 4 neumaticos mas un parchado de la rueda de repuesto?")
print(respuesta)


🙋 Cliente: ¿Cuanto seria un cambio e 4 neumaticos mas un parchado de la rueda de repuesto?

🔧 Vulcanizador (RAG):
Diagnóstico: El cliente solicita el cambio de 4 neumáticos y además requiere un parche interno vulcanizado para la rueda de repuesto (pinchazo estándar).

Recomendación: Realizar el cambio de los 4 neumáticos, aprovechando la promoción que incluye montaje y balanceo sin costo. Adicionalmente, aplicar un parche interno vulcanizado a la rueda de repuesto.

Precio estimado:
- Cambio de 4 neumáticos: Montaje + balanceo incluido sin costo (solo se cobra el valor de los neumáticos, no especificado en el contexto).
- Parche interno vulcanizado (pinchazo estándar): $6.000

Total estimado por servicios: $6.000  
*Si la orden supera los $100.000 (considerando el valor de los neumáticos), se aplica 15% de descuento en el servicio más caro (en este caso, el parche: $6.000 - 15% = $5.100).

Urgencia: Moderada. El cambio de neumáticos y la reparación de la rueda de repuesto son importan